# 🛒 E-Commerce Sales Analysis Using PySpark

Welcome to this end-to-end PySpark Data Engineering and Analytics project. In this notebook, we build an enterprise-grade lakehouse pipeline to analyze large-scale e-commerce transaction data.

### 📌 Key Project Objectives:
1. **Production Spark Configuration**: Setting up Adaptive Query Execution (AQE), Spark SQL parameters, and memory tuning.
2. **Data Ingestion & Bronze Layer**: Reading raw CSV datasets using explicit schemas (`StructType`, `StructField`) to prevent expensive inference passes.
3. **Data Cleansing & Silver Layer**: Filtering nulls, invalid states, derived date dimensions, and caching.
4. **Executive Financial KPIs**: GMV, Net Sales, Average Order Value (AOV), and customer order metrics.
5. **Time-Series & MoM Growth**: Using Spark Window lag functions to compute monthly growth rates and peak shopping hours.
6. **Product & Category Intelligence**: Top products, category share of revenue, and margin analysis.
7. **Customer RFM Segmentation**: Quantile-based Recency, Frequency, Monetary (RFM) modeling using `ntile(5)`.
8. **Cohort Retention Analysis**: Customer lifetime behavior tracking across acquisition months.
9. **Spark Optimization Best Practices**: Broadcast joins, query execution plans (`explain()`), and Parquet partitioning.

In [ ]:
import os
import sys
if os.name == 'nt' and os.path.exists(r'C:\hadoop'):
    os.environ['HADOOP_HOME'] = r'C:\hadoop'
    os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

# Initialize Spark Session with AQE and local core allocation
spark = (
    SparkSession.builder
    .appName("Ecommerce_Sales_Analysis_Notebook")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session Active! Version: {spark.version}")

## 1. Schema Definition & Data Ingestion (Bronze Layer)

In [ ]:
transaction_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("customer_tier", StringType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_percent", DoubleType(), True),
    StructField("line_total", DoubleType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("shipping_cost", DoubleType(), True)
])

df_raw = spark.read.option("header", "true").schema(transaction_schema).csv("data/ecommerce_transactions.csv")
print(f"Total Ingested Rows: {df_raw.count():,}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

## 2. Data Cleaning & Feature Engineering (Silver Layer)

In [ ]:
# Filter out bad records, calculate date dimensions and sales metrics
df_silver = (
    df_raw
    .dropna(subset=["order_id", "customer_id", "product_id", "line_total", "order_date"])
    .filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_year_month", F.date_format("order_date", "yyyy-MM"))
    .withColumn("order_day_name", F.date_format("order_date", "EEEE"))
    .withColumn("order_hour", F.hour("order_date"))
    .withColumn("gross_sales", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("discount_amount", F.round(F.col("gross_sales") * F.col("discount_percent"), 2))
)

df_silver.cache()
print(f"Silver Layer Rows: {df_silver.count():,}")
df_silver.createOrReplaceTempView("transactions")

## 3. Executive Sales KPIs
Filtering for completed and shipped orders to calculate realized revenues.

In [ ]:
df_realized = df_silver.filter(F.col("order_status").isin("Completed", "Shipped"))

kpi_df = df_realized.agg(
    F.round(F.sum("gross_sales"), 2).alias("Gross_Sales_USD"),
    F.round(F.sum("discount_amount"), 2).alias("Total_Discounts_USD"),
    F.round(F.sum("line_total"), 2).alias("Net_Revenue_USD"),
    F.countDistinct("order_id").alias("Total_Orders"),
    F.countDistinct("customer_id").alias("Unique_Customers"),
    F.sum("quantity").alias("Total_Units_Sold"),
    F.round(F.sum("line_total") / F.countDistinct("order_id"), 2).alias("AOV_Avg_Order_Value"),
    F.round(F.sum("quantity") / F.countDistinct("order_id"), 2).alias("Avg_Items_Per_Order")
)
kpi_df.show(truncate=False)

## 4. Month-over-Month (MoM) Growth Analysis Using Window Functions

In [ ]:
monthly_sales = (
    df_realized
    .groupBy("order_year_month")
    .agg(
        F.round(F.sum("line_total"), 2).alias("monthly_revenue"),
        F.countDistinct("order_id").alias("orders"),
        F.countDistinct("customer_id").alias("active_customers")
    )
)

month_win = Window.orderBy("order_year_month")
monthly_growth = (
    monthly_sales
    .withColumn("prev_month_revenue", F.lag("monthly_revenue", 1).over(month_win))
    .withColumn(
        "mom_growth_pct",
        F.round(((F.col("monthly_revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue")) * 100, 2)
    )
    .orderBy("order_year_month")
)
monthly_growth.show(24, truncate=False)

# Convert to Pandas for plotting
pdf_monthly = monthly_growth.toPandas()
plt.figure(figsize=(12, 4))
plt.plot(pdf_monthly["order_year_month"], pdf_monthly["monthly_revenue"], marker="o", color="#1f77b4", linewidth=2)
plt.title("Monthly Net Revenue Trend (USD)", fontsize=14, fontweight="bold")
plt.xlabel("Year-Month")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Category Share & Top Products (Pareto Analysis)

In [ ]:
total_win = Window.partitionBy()

cat_summary = (
    df_realized
    .groupBy("category")
    .agg(
        F.round(F.sum("line_total"), 2).alias("revenue"),
        F.sum("quantity").alias("units")
    )
    .withColumn("total_rev", F.sum("revenue").over(total_win))
    .withColumn("share_pct", F.round((F.col("revenue") / F.col("total_rev")) * 100, 2))
    .drop("total_rev")
    .orderBy(F.desc("revenue"))
)
cat_summary.show(truncate=False)

# Top 10 Best Selling Products
rank_win = Window.orderBy(F.desc("product_revenue"))
top_prods = (
    df_realized
    .groupBy("product_name", "category")
    .agg(F.round(F.sum("line_total"), 2).alias("product_revenue"))
    .withColumn("rank", F.dense_rank().over(rank_win))
    .filter(F.col("rank") <= 10)
    .orderBy("rank")
)
top_prods.show(10, truncate=False)

## 6. Customer RFM (Recency, Frequency, Monetary) Segmentation

In [ ]:
max_date = df_realized.select(F.max("order_date")).collect()[0][0]

rfm_raw = (
    df_realized
    .groupBy("customer_id", "customer_name")
    .agg(
        F.datediff(F.lit(max_date), F.max("order_date")).alias("recency_days"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(F.sum("line_total"), 2).alias("monetary")
    )
)

# Ntile scoring
r_w = Window.orderBy(F.desc("recency_days"))
f_w = Window.orderBy("frequency")
m_w = Window.orderBy("monetary")

rfm_segmented = (
    rfm_raw
    .withColumn("r_score", F.ntile(5).over(r_w))
    .withColumn("f_score", F.ntile(5).over(f_w))
    .withColumn("m_score", F.ntile(5).over(m_w))
    .withColumn(
        "segment",
        F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
         .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3), "Loyal Customers")
         .when((F.col("r_score") >= 4) & (F.col("f_score") <= 2), "Promising / New Customers")
         .when((F.col("r_score") <= 2) & (F.col("f_score") >= 3), "At Risk")
         .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2), "Lost Customers")
         .otherwise("Potential Loyalists")
    )
)

rfm_summary = (
    rfm_segmented
    .groupBy("segment")
    .agg(
        F.count("customer_id").alias("customers"),
        F.round(F.sum("monetary"), 2).alias("segment_revenue"),
        F.round(F.avg("monetary"), 2).alias("avg_spend"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency")
    )
    .orderBy(F.desc("segment_revenue"))
)
rfm_summary.show(truncate=False)

## 7. Cohort Retention Matrix

In [ ]:
cust_first_order = Window.partitionBy("customer_id")
df_cohort = (
    df_realized
    .withColumn("first_order_date", F.min("order_date").over(cust_first_order))
    .withColumn("cohort_month", F.date_format("first_order_date", "yyyy-MM"))
    .withColumn(
        "cohort_index",
        (F.year("order_date") - F.year("first_order_date")) * 12 + (F.month("order_date") - F.month("first_order_date"))
    )
)

cohort_matrix = (
    df_cohort
    .filter(F.col("cohort_index") <= 6)
    .groupBy("cohort_month")
    .pivot("cohort_index", [0, 1, 2, 3, 4, 5, 6])
    .agg(F.countDistinct("customer_id"))
    .orderBy("cohort_month")
)
cohort_matrix.show(15, truncate=False)

## 8. Export Optimized Parquet Gold Table

In [ ]:
gold_path = "output/gold_sales_partitioned"
(
    df_silver
    .write
    .mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(gold_path)
)
print(f"Successfully written partitioned Parquet files to: {gold_path}")